# Mondrian conformal HDF5 demo

This notebook runs Mondrian conformal analysis using saved CNN predictions, labels, and embeddings.

It supports two modes:

- real calibration/test mode, using `pred_cal`, `y_cal`, `emb_cal`, `pred_test`, `y_test`, and `emb_test`
- debug mode, where validation predictions are split into pseudo-calibration and pseudo-test subsets

Debug mode is useful for testing the pipeline, but must not be reported as final conformal performance.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import iqr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Locate project root robustly from the current notebook location
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "cbc_pe" / "src").exists():
    PROJECT_ROOT = cwd / "cbc_pe"
else:
    raise RuntimeError(
        f"Could not locate project root from cwd={cwd}. "
        "Expected to find a 'src/' directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")
DATA_RESULTS = DATA_ROOT / "results"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
RESULTS_DIR = DATA_RESULTS / dataset_id

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

## Run Selection

For the 100k architecture-search setup, calibration and test sets are not available. Use debug mode only for pipeline development.

For final reporting, use a prediction file produced from a 70/10/10/10 or similar train/validation/calibration/test split.

In [ ]:
# ---------------------------------------------------------------------
# Run selection
# ---------------------------------------------------------------------

RUN_ID = "500k_M00_baseline_emb64_seed123"
# RUN_ID = "100k_M00_baseline_emb64_seed123"
# RUN_ID = "100k_M00_baseline_emb64_seed124"
# RUN_ID = "100k_M04_pooldeep_emb128_pool4"

DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")
DATA_RESULTS = DATA_ROOT / "results"

runs = {
    "500k_M00_baseline_emb64_seed123": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
            "_SimpleCNN_Baseline_simple_emb64_mse_MSELoss_seed123"
            "_train_val_cal_test_predictions_embeddings.npz"
        ),
    },
    "100k_M00_baseline_emb64_seed123": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
            "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed123"
            "_train_val_predictions_embeddings.npz"
        ),
    },
    "100k_M00_baseline_emb64_seed124": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
            "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed124"
            "_train_val_predictions_embeddings.npz"
        ),
    },
    "100k_M04_pooldeep_emb128_pool4": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
            "_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_MSELoss_seed123"
            "_train_val_predictions_embeddings.npz"
        ),
    },
}

run = runs[RUN_ID]

dataset_id = run["dataset_id"]
RESULTS_DIR = DATA_RESULTS / dataset_id
prediction_file = RESULTS_DIR / run["prediction_filename"]

assert prediction_file.exists(), prediction_file

print("RUN_ID:", RUN_ID)
print("dataset_id:", dataset_id)
print("RESULTS_DIR:", RESULTS_DIR)
print("prediction_file:", prediction_file)

data = np.load(prediction_file, allow_pickle=True)
print("NPZ keys:", data.files)

## Build calibration and test arrays

In [ ]:
def load_conformal_arrays(
    data,
    debug_split_val=False,
    debug_cal_fraction=0.5,
    debug_seed=123,
):
    """
    Load arrays needed for Mondrian/conformal.

    Preferred real mode:
        pred_cal, y_cal, emb_cal
        pred_test, y_test, emb_test

    Debug mode:
        If real cal/test are not available, split val into pseudo_cal/pseudo_test.
        This is only for debugging the notebook, not for final reporting.
    """

    files = set(data.files)

    has_real_cal_test = {
        "pred_cal",
        "y_cal",
        "emb_cal",
        "pred_test",
        "y_test",
        "emb_test",
    }.issubset(files)

    if has_real_cal_test:
        print("Using real cal/test arrays.")

        pred_cal = data["pred_cal"]
        y_cal = data["y_cal"]
        emb_cal = data["emb_cal"]

        pred_test = data["pred_test"]
        y_test = data["y_test"]
        emb_test = data["emb_test"]

        idx_cal = data["idx_cal"] if "idx_cal" in files else None
        idx_test = data["idx_test"] if "idx_test" in files else None

        mode = "real_cal_test"

    else:
        if not debug_split_val:
            raise KeyError(
                "No real cal/test arrays found in prediction file. "
                "Expected pred_cal/y_cal/emb_cal and pred_test/y_test/emb_test. "
                "For notebook debugging only, set debug_split_val=True."
            )

        required_val = {"pred_val", "y_val", "emb_val"}

        if not required_val.issubset(files):
            raise KeyError(
                "Cannot create debug split because pred_val/y_val/emb_val are missing."
            )

        print("WARNING: using validation split as pseudo cal/test.")
        print("This is only for debugging. Do not report these results as final.")

        pred_val = data["pred_val"]
        y_val = data["y_val"]
        emb_val = data["emb_val"]

        n_val = pred_val.shape[0]
        rng = np.random.default_rng(debug_seed)
        perm = rng.permutation(n_val)

        n_cal = int(debug_cal_fraction * n_val)

        cal_local = perm[:n_cal]
        test_local = perm[n_cal:]

        pred_cal = pred_val[cal_local]
        y_cal = y_val[cal_local]
        emb_cal = emb_val[cal_local]

        pred_test = pred_val[test_local]
        y_test = y_val[test_local]
        emb_test = emb_val[test_local]

        if "idx_val" in files:
            idx_val = data["idx_val"]
            idx_cal = idx_val[cal_local]
            idx_test = idx_val[test_local]
        else:
            idx_cal = cal_local
            idx_test = test_local

        mode = "debug_val_split"

    y_mean = data["y_mean"]
    y_std = data["y_std"]

    if "label_names" in files:
        label_names = data["label_names"].tolist()
    else:
        label_names = ["chirp_mass", "total_mass", "chi_eff"]

    return {
        "mode": mode,
        "pred_cal": pred_cal,
        "y_cal": y_cal,
        "emb_cal": emb_cal,
        "pred_test": pred_test,
        "y_test": y_test,
        "emb_test": emb_test,
        "idx_cal": idx_cal,
        "idx_test": idx_test,
        "y_mean": y_mean,
        "y_std": y_std,
        "label_names": label_names,
    }

In [ ]:
DEBUG_SPLIT_VAL = False  # Debug only. Do not report as final conformal performance.

arrays = load_conformal_arrays(
    data,
    debug_split_val=DEBUG_SPLIT_VAL,
    debug_cal_fraction=0.5,
    debug_seed=123,
)

mode = arrays["mode"]

pred_cal = arrays["pred_cal"]
y_cal = arrays["y_cal"]
emb_cal = arrays["emb_cal"]

pred_test = arrays["pred_test"]
y_test = arrays["y_test"]
emb_test = arrays["emb_test"]

idx_cal = arrays["idx_cal"]
idx_test = arrays["idx_test"]

y_mean = arrays["y_mean"]
y_std = arrays["y_std"]
label_names = arrays["label_names"]

print("mode:", mode)
print("pred_cal:", pred_cal.shape)
print("y_cal:", y_cal.shape)
print("emb_cal:", emb_cal.shape)
print("pred_test:", pred_test.shape)
print("y_test:", y_test.shape)
print("emb_test:", emb_test.shape)
print("y_mean:", y_mean)
print("y_std:", y_std)
print("label_names:", label_names)

if mode != "real_cal_test":
    print("WARNING: running in debug pseudo-cal/test mode. Do not report as final conformal performance.")

In [ ]:
# Sanity checks

assert pred_cal.ndim == 2
assert pred_test.ndim == 2
assert y_cal.ndim == 2
assert y_test.ndim == 2

assert pred_cal.shape == y_cal.shape
assert pred_test.shape == y_test.shape

assert pred_cal.shape[1] == len(label_names)
assert pred_test.shape[1] == len(label_names)

assert emb_cal.ndim == 2
assert emb_test.ndim == 2

assert emb_cal.shape[0] == pred_cal.shape[0]
assert emb_test.shape[0] == pred_test.shape[0]

assert y_mean.shape[0] == len(label_names)
assert y_std.shape[0] == len(label_names)
assert np.all(y_std > 0)

for name, arr in {
    "pred_cal": pred_cal,
    "y_cal": y_cal,
    "pred_test": pred_test,
    "y_test": y_test,
    "emb_cal": emb_cal,
    "emb_test": emb_test,
    "y_mean": y_mean,
    "y_std": y_std,
}.items():
    assert np.all(np.isfinite(arr)), f"{name} contains NaN or inf"

if mode != "real_cal_test":
    print("WARNING: notebook is running in debug mode:", mode)
    print("Do not use these results as final conformal/Mondrian results.")

print("All sanity checks passed.")

In [ ]:
def inverse_standardize(y_std_values, y_mean, y_std):
    return y_std_values * y_std + y_mean


y_cal_phys = inverse_standardize(y_cal, y_mean, y_std)
y_test_phys = inverse_standardize(y_test, y_mean, y_std)

pred_cal_phys = inverse_standardize(pred_cal, y_mean, y_std)
pred_test_phys = inverse_standardize(pred_test, y_mean, y_std)

label_ranges_phys = {
    label: np.max(y_test_phys[:, j]) - np.min(y_test_phys[:, j])
    for j, label in enumerate(label_names)
}

label_ranges_phys

## Start Mondrian: Build the DF

In [ ]:
from src.conformal.pipeline import run_mondrian_regression

confidence_level = 0.90

if mode == "real_cal_test":
    n_bins_grid = [4, 6, 8, 12, 16, 24, 32, 48]
else:
    n_bins_grid = [4, 6, 8, 12, 16, 24]

taxonomy_modes = ["prediction", "difficulty"]
interval_modes = ["symmetric", "asymmetric"]

n_neighbors = 5
min_samples_per_bin = 20 if mode == "real_cal_test" else 10

rows = []
all_results = {}



for taxonomy_mode in taxonomy_modes:
    for interval_mode in interval_modes:
        for n_bins in n_bins_grid:

            kwargs = dict(
                pred_cal=pred_cal,
                pred_test=pred_test,
                y_cal=y_cal,
                y_test=y_test,
                n_bins=n_bins,
                confidence_level=confidence_level,
                apply_jitter=True,
                interval_mode=interval_mode,
                taxonomy_mode=taxonomy_mode,
                min_samples_per_bin=min_samples_per_bin,
                tolerance_sigmas=(1, 2, 3),
            )

            if taxonomy_mode == "difficulty":
                kwargs.update(
                    cal_embedding=emb_cal,
                    target_embedding=emb_test,
                    n_neighbors=n_neighbors,
                )

            result = run_mondrian_regression(**kwargs)
            all_results[(taxonomy_mode, interval_mode, n_bins)] = result

            metrics = result.metrics

            widths_std = result.upper - result.lower
            widths_phys = widths_std * y_std

            for j, label in enumerate(label_names):
                # ------------------------------------------------------------
                # Existing p-value based local undercoverage diagnostic
                # ------------------------------------------------------------
                n_bad_bins_p005 = int(
                    np.nansum(metrics["bin_undercoverage_pvalue"][:, j] < 0.05)
                )

                # ------------------------------------------------------------
                # New: local 2-sigma bin-wise validity diagnostics
                # ------------------------------------------------------------
                coverage_bin = metrics["coverage_per_bin"][:, j]
                count_bin = metrics["counts_per_bin"][:, j]

                bin_tol = metrics["bin_tolerance_normal"]

                bin_2sigma_low = bin_tol["2sigma_low"][:, j]
                bin_2sigma_high = bin_tol["2sigma_high"][:, j]
                bin_2sigma_width = bin_tol["2sigma_width"][:, j]

                valid_bins = count_bin > 0

                bin_within_2sigma = (
                    (coverage_bin >= bin_2sigma_low)
                    & (coverage_bin <= bin_2sigma_high)
                    & valid_bins
                )

                bin_under_2sigma = (
                    (coverage_bin < bin_2sigma_low)
                    & valid_bins
                )

                bin_over_2sigma = (
                    (coverage_bin > bin_2sigma_high)
                    & valid_bins
                )

                all_bins_within_2sigma = bool(np.all(bin_within_2sigma[valid_bins]))

                n_bins_outside_2sigma = int(np.sum(~bin_within_2sigma[valid_bins]))
                outside_bin_fraction_2sigma = float(np.mean(~bin_within_2sigma[valid_bins]))

                n_bins_under_2sigma = int(np.sum(bin_under_2sigma[valid_bins]))
                under_bin_fraction_2sigma = float(np.mean(bin_under_2sigma[valid_bins]))

                n_bins_over_2sigma = int(np.sum(bin_over_2sigma[valid_bins]))
                over_bin_fraction_2sigma = float(np.mean(bin_over_2sigma[valid_bins]))

                min_bin_2sigma_low = float(np.nanmin(bin_2sigma_low[valid_bins]))
                max_bin_2sigma_high = float(np.nanmax(bin_2sigma_high[valid_bins]))
                median_bin_2sigma_width = float(np.nanmedian(bin_2sigma_width[valid_bins]))

                row = {
                    "mode": mode,
                    "taxonomy_mode": taxonomy_mode,
                    "interval_mode": interval_mode,
                    "n_bins": n_bins,
                    "label": label,
                    "label_index": j,

                    "n_samples": int(metrics["n_samples_per_label"][j]),
                    "covered_count": int(metrics["covered_count_global"][j]),

                    "global_coverage": metrics["global_coverage"][j],
                    "miscoverage": metrics["miscoverage"][j],
                    "global_coverage_gap": metrics["global_coverage_gap"][j],
                    "global_undercoverage_pvalue": metrics["global_undercoverage_pvalue"][j],

                    "global_mean_width_std": metrics["global_mean_width"][j],
                    "global_median_width_std": metrics["global_median_width"][j],

                    "global_mean_width_phys": np.mean(widths_phys[:, j]),
                    "global_median_width_phys": np.median(widths_phys[:, j]),
                    "global_iqr_width_phys": iqr(widths_phys[:, j]),

                    "normalized_median_width_range": (
                        np.median(widths_phys[:, j]) / label_ranges_phys[label]
                    ),

                    "min_coverage_per_bin": metrics["min_coverage_per_label"][j],
                    "max_undercoverage_gap": metrics["max_undercoverage_gap"][j],

                    "min_count_per_bin": int(np.nanmin(metrics["counts_per_bin"][:, j])),
                    "max_count_per_bin": int(np.nanmax(metrics["counts_per_bin"][:, j])),

                    "n_bad_bins_p005": n_bad_bins_p005,
                    "bad_bin_fraction": n_bad_bins_p005 / n_bins,

                    # Local 2-sigma bin-wise validity
                    "all_bins_within_2sigma": all_bins_within_2sigma,
                    "n_bins_outside_2sigma": n_bins_outside_2sigma,
                    "outside_bin_fraction_2sigma": outside_bin_fraction_2sigma,

                    "n_bins_under_2sigma": n_bins_under_2sigma,
                    "under_bin_fraction_2sigma": under_bin_fraction_2sigma,

                    "n_bins_over_2sigma": n_bins_over_2sigma,
                    "over_bin_fraction_2sigma": over_bin_fraction_2sigma,

                    "min_bin_2sigma_low": min_bin_2sigma_low,
                    "max_bin_2sigma_high": max_bin_2sigma_high,
                    "median_bin_2sigma_width": median_bin_2sigma_width,

                    "global_lower_miss_rate": metrics["global_lower_miss_rate"][j],
                    "global_upper_miss_rate": metrics["global_upper_miss_rate"][j],
                    "global_tail_miss_imbalance": metrics["global_tail_miss_imbalance"][j],
                }

                global_tol = metrics["global_tolerance_normal"]

                for k in [1, 2, 3]:
                    low = global_tol[f"{k}sigma_low"][j]
                    high = global_tol[f"{k}sigma_high"][j]
                    width = global_tol[f"{k}sigma_width"][j]

                    row[f"global_tol_{k}sigma_low"] = low
                    row[f"global_tol_{k}sigma_high"] = high
                    row[f"global_tol_{k}sigma_width"] = width
                    row[f"global_within_{k}sigma"] = bool(low <= metrics["global_coverage"][j] <= high)

                rows.append(row)

summary_df = pd.DataFrame(rows)


print("summary_df shape:", summary_df.shape)
summary_df.head()



In [ ]:
column_descriptions = {
    # Identity / configuration
    "mode": "Evaluation mode, e.g. real_cal_test.",
    "taxonomy_mode": "Mondrian taxonomy used to define bins: prediction or difficulty.",
    "interval_mode": "Conformal interval type: symmetric or asymmetric.",
    "n_bins": "Number of Mondrian bins.",
    "label": "Target label: chirp_mass, total_mass, chi_eff.",
    "label_index": "Index of the target label.",

    # Global coverage
    "n_samples": "Number of test samples evaluated.",
    "covered_count": "Number of test samples whose true value is inside the interval.",
    "global_coverage": "Empirical coverage on the test set.",
    "miscoverage": "1 - global_coverage.",
    "global_coverage_gap": "Absolute gap between empirical coverage and target coverage.",
    "global_undercoverage_pvalue": "Binomial p-value for global undercoverage.",

    # Widths
    "global_mean_width_std": "Mean interval width in standardized target units.",
    "global_median_width_std": "Median interval width in standardized target units.",
    "global_mean_width_phys": "Mean interval width in physical units.",
    "global_median_width_phys": "Median interval width in physical units.",
    "global_iqr_width_phys": "IQR of interval widths in physical units.",
    "normalized_median_width_range": "Median physical width normalized by the label physical range.",

    # Local bin summary
    "min_coverage_per_bin": "Minimum empirical coverage among bins.",
    "max_undercoverage_gap": "Largest undercoverage gap among bins relative to target coverage.",
    "min_count_per_bin": "Minimum number of test samples in any bin.",
    "max_count_per_bin": "Maximum number of test samples in any bin.",
    "n_bad_bins_p005": "Number of bins flagged as significantly undercovered at p < 0.05.",
    "bad_bin_fraction": "Fraction of bins flagged as significantly undercovered.",

    # Tail balance
    "global_lower_miss_rate": "Fraction of samples missed below the lower bound.",
    "global_upper_miss_rate": "Fraction of samples missed above the upper bound.",
    "global_tail_miss_imbalance": "Absolute imbalance between lower and upper miss rates.",

    # Global binomial tolerance
    "global_tol_1sigma_low": "Lower 1σ tolerance for global coverage.",
    "global_tol_1sigma_high": "Upper 1σ tolerance for global coverage.",
    "global_tol_1sigma_width": "Half-width of 1σ global tolerance.",
    "global_within_1sigma": "Whether global coverage is inside 1σ tolerance.",
    "global_tol_2sigma_low": "Lower 2σ tolerance for global coverage.",
    "global_tol_2sigma_high": "Upper 2σ tolerance for global coverage.",
    "global_tol_2sigma_width": "Half-width of 2σ global tolerance.",
    "global_within_2sigma": "Whether global coverage is inside 2σ tolerance.",
    "global_tol_3sigma_low": "Lower 3σ tolerance for global coverage.",
    "global_tol_3sigma_high": "Upper 3σ tolerance for global coverage.",
    "global_tol_3sigma_width": "Half-width of 3σ global tolerance.",
    "global_within_3sigma": "Whether global coverage is inside 3σ tolerance.",
}

In [ ]:
new_local_cols = [
    "all_bins_within_2sigma",
    "n_bins_outside_2sigma",
    "outside_bin_fraction_2sigma",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "n_bins_over_2sigma",
    "over_bin_fraction_2sigma",
    "min_bin_2sigma_low",
    "max_bin_2sigma_high",
    "median_bin_2sigma_width",
]

summary_df[
    [
        "label",
        "taxonomy_mode",
        "interval_mode",
        "n_bins",
        "global_coverage",
        "global_within_2sigma",
        "min_coverage_per_bin",
        "max_undercoverage_gap",
    ]
    + new_local_cols
].head(20)

## Save the results

In [ ]:
mondrian_results_dir = RESULTS_DIR / "mondrian"
mondrian_results_dir.mkdir(parents=True, exist_ok=True)

prediction_stem = prediction_file.stem

summary_path = mondrian_results_dir / f"{prediction_stem}_mondrian_summary_{mode}.csv"

summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)

## Ranking Configurations

### Filtering the configurations

In [ ]:
# Final Mondrian configuration selection policy
#
# For chirp_mass and total_mass, we require strong local validity.
# For chi_eff, if no configuration satisfies the strong criterion, we use a relaxed
# best-available policy and report it explicitly.

WIDTH_TOLERANCE = 1.05
MIN_COUNT_PER_BIN = 100
top_k = 5

strong_labels = ["chirp_mass", "total_mass"]

required_cols = [
    "label",
    "global_within_2sigma",
    "n_bins_under_2sigma",
    "n_bad_bins_p005",
    "min_count_per_bin",
    "global_median_width_phys",
    "n_bins",
    "n_bins_outside_2sigma",
    "bad_bin_fraction",
    "max_undercoverage_gap",
    "global_tail_miss_imbalance",
]

missing_cols = [col for col in required_cols if col not in summary_df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in summary_df: {missing_cols}")


def get_label_candidates(summary_df, label):
    """Return candidate configurations and the selection policy used for one label."""
    base = summary_df[
        (summary_df["label"] == label)
        & (summary_df["global_within_2sigma"])
        & (summary_df["min_count_per_bin"] >= MIN_COUNT_PER_BIN)
    ].copy()

    if base.empty:
        return base, "no_global_valid_candidates"

    strict = base[
        (base["n_bad_bins_p005"] == 0)
        & (base["n_bins_under_2sigma"] == 0)
        & (base["n_bins_outside_2sigma"] == 0)
    ].copy()

    if not strict.empty:
        return strict, "strict_local_validity"

    strong = base[
        (base["n_bad_bins_p005"] == 0)
        & (base["n_bins_under_2sigma"] == 0)
    ].copy()

    if not strong.empty:
        return strong, "strong_undercoverage_safe"

    return base, "relaxed_best_available"

In [ ]:
candidate_tables = []
selected_rows = []

for label in label_names:
    candidates, policy = get_label_candidates(summary_df, label)

    if candidates.empty:
        print(f"No valid candidates found for {label} under policy: {policy}")
        continue

    candidates = candidates.copy()
    candidates["selection_policy"] = policy

    if policy in ["strict_local_validity", "strong_undercoverage_safe"]:
        min_width = candidates["global_median_width_phys"].min()
        width_threshold = WIDTH_TOLERANCE * min_width

        candidates = candidates[
            candidates["global_median_width_phys"] <= width_threshold
        ].copy()

        candidates["min_width_for_label"] = min_width
        candidates["width_threshold"] = width_threshold
        candidates["relative_width_excess"] = (
            candidates["global_median_width_phys"] / min_width - 1.0
        )

        ranked = candidates.sort_values(
            by=[
                "n_bins_outside_2sigma",
                "max_undercoverage_gap",
                "global_median_width_phys",
                "n_bins",
                "global_tail_miss_imbalance",
            ],
            ascending=[
                True,
                True,
                True,
                False,
                True,
            ],
        )

    else:
        # Relaxed fallback:
        # Prioritize statistical safety first, then interval width, then adaptivity.
        MAX_BAD_BINS_RELAXED = 1
        MAX_UNDER_BINS_RELAXED = 1

        relaxed_safe = candidates[
            (candidates["n_bad_bins_p005"] <= MAX_BAD_BINS_RELAXED)
            & (candidates["n_bins_under_2sigma"] <= MAX_UNDER_BINS_RELAXED)
        ].copy()

        if not relaxed_safe.empty:
            candidates = relaxed_safe

        min_width = candidates["global_median_width_phys"].min()
        candidates["min_width_for_label"] = min_width
        candidates["width_threshold"] = np.nan
        candidates["relative_width_excess"] = (
            candidates["global_median_width_phys"] / min_width - 1.0
        )

        ranked = candidates.sort_values(
            by=[
                "n_bins_under_2sigma",
                "n_bins_outside_2sigma",
                "n_bad_bins_p005",
                "max_undercoverage_gap",
                "global_median_width_phys",
                "n_bins",
            ],
            ascending=[
                True,
                True,
                True,
                True,
                True,
                False,
            ],
        )

    candidate_tables.append(ranked.head(top_k))
    selected_rows.append(ranked.iloc[0])

top_by_label = pd.concat(candidate_tables, ignore_index=True)
final_by_label = pd.DataFrame(selected_rows)

In [ ]:
display_cols = [
    "label",
    "selection_policy",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "all_bins_within_2sigma",
    "n_bins_outside_2sigma",
    "n_bins_under_2sigma",
    "n_bad_bins_p005",
    "bad_bin_fraction",
    "min_coverage_per_bin",
    "max_undercoverage_gap",
    "min_count_per_bin",
    "global_median_width_phys",
    "min_width_for_label",
    "relative_width_excess",
    "global_tail_miss_imbalance",
]

top_by_label[display_cols]

In [ ]:
# Final selected configuration per label

final_by_label[display_cols]

In [ ]:
# 1. First we remove the configs that does not pass the pre-cut:
# within 2 sigma nominal CL, min counts per bin is 20 and fraction of bins with statisticall p<5% is less or equal 10 %
if mode == "real_cal_test":
    min_count_threshold = 100
    max_bad_bin_fraction = 0.10
else:
    min_count_threshold = 20
    max_bad_bin_fraction = 0.10

candidate_df = summary_df[
    (summary_df["global_within_2sigma"]) &
    (summary_df["n_bins_under_2sigma"] == 0) &
    (summary_df["min_count_per_bin"] >= min_count_threshold) 
].copy()

ranking_df = candidate_df.sort_values(
    [
        "label",
        "bad_bin_fraction",
        "n_bins",
        "max_undercoverage_gap",
        "global_median_width_std",
        
         
    ],
    ascending=[True, True, False, True, True],
)

ranking_df[
    [
        "label",
        "taxonomy_mode",
        "interval_mode",
        "n_bins",
        "global_coverage",
        "global_within_1sigma",
        "global_within_2sigma",
        "global_undercoverage_pvalue",
        "min_coverage_per_bin",
        "max_undercoverage_gap",
        "n_bad_bins_p005",
        "bad_bin_fraction",
        "global_median_width_std",
        "global_median_width_phys",
        "normalized_median_width_range",
        "global_iqr_width_phys",
        "global_lower_miss_rate",
        "global_upper_miss_rate",
        "global_tail_miss_imbalance",
        "min_count_per_bin",
    ]
]

print("Mode:", mode)
print("Summary configs:", summary_df.groupby("label").size())
print("Passed configs:", candidate_df.groupby("label").size())


## Criterio de selección de la configuración Mondrian final

La selección de la configuración Mondrian final se plantea como un problema multiobjetivo. El objetivo no es únicamente minimizar la anchura de los intervalos, ni tampoco maximizar el número de bins de forma aislada. La motivación de usar Mondrian conformal es introducir adaptabilidad local mediante bins, pero dicha adaptabilidad sólo es útil si no degrada la validez estadística de los intervalos.

Por tanto, la política de selección combina tres objetivos:

1. mantener una cobertura global compatible con el nivel nominal;
2. evitar problemas de infra-cobertura local en los bins;
3. obtener intervalos lo más estrechos posible, favoreciendo configuraciones con mayor número de bins sólo cuando las métricas de validez local siguen siendo razonables.

Se define primero un conjunto base de configuraciones candidatas que cumplen:

* cobertura global dentro de la banda nominal de 2σ;
* al menos 100 muestras por bin;
* evaluación independiente con calibración y test reales.

A partir de este conjunto base se aplica una política jerárquica de validez local.

### Nivel 1: validez local estricta

Una configuración se considera de validez local estricta (`strict_local_validity`) si cumple:

* `global_within_2sigma == True`;
* `n_bad_bins_p005 == 0`;
* `n_bins_under_2sigma == 0`;
* `n_bins_outside_2sigma == 0`;
* `min_count_per_bin >= 100`.

Esto exige que no haya bins con infra-cobertura estadísticamente significativa según el test binomial, que ningún bin caiga por debajo de la banda local de 2σ, y que todos los bins estén dentro de la banda local de 2σ.

Dentro de las configuraciones estrictamente válidas, se permite una tolerancia de anchura del 5% respecto a la configuración válida más estrecha de cada parámetro. Entre esas configuraciones, se prioriza primero la calidad local, después la anchura física mediana, y finalmente el número de bins. De este modo se evita escoger configuraciones más adaptativas si esa adaptabilidad introduce degradación local o intervalos innecesariamente más anchos.

### Nivel 2: configuración relajada disponible

Si no existe una configuración que satisfaga la validez local estricta, se aplica una política relajada (`relaxed_best_available`). Esta política se utiliza principalmente para parámetros más difíciles, como `chi_eff`.

En este caso se mantiene la exigencia de cobertura global dentro de 2σ y tamaño mínimo de bin, pero se permite que exista algún bin problemático. La selección se hace minimizando, por orden de prioridad:

* número de bins por debajo de 2σ local;
* número de bins fuera de 2σ local;
* número de bins con p-value de infra-cobertura menor que 0.05;
* máximo gap de infra-cobertura;
* anchura física mediana;
* número de bins.

Esta política permite reportar explícitamente que el parámetro no alcanza el mismo nivel de validez local que los demás, sin ocultar el problema ni forzar artificialmente una configuración que no cumple los criterios fuertes.

## Resultados observados

Para `chirp_mass`, la mejor configuración seleccionada cumple validez local estricta:

* taxonomía: `difficulty`;
* intervalo: `asymmetric`;
* número de bins: 8;
* cobertura global: 0.898967;
* anchura física mediana: 14.712123;
* `n_bad_bins_p005 = 0`;
* `n_bins_under_2sigma = 0`;
* `n_bins_outside_2sigma = 0`;
* `all_bins_within_2sigma = True`.

Esta configuración proporciona una cobertura global compatible con el nivel nominal y no presenta bins localmente problemáticos según los criterios considerados.

Para `total_mass`, también se encuentra una configuración con validez local estricta:

* taxonomía: `difficulty`;
* intervalo: `asymmetric`;
* número de bins: 4;
* cobertura global: 0.902400;
* anchura física mediana: 28.008818;
* `n_bad_bins_p005 = 0`;
* `n_bins_under_2sigma = 0`;
* `n_bins_outside_2sigma = 0`;
* `all_bins_within_2sigma = True`.

Aunque existen configuraciones con más bins, la configuración seleccionada ofrece mejor equilibrio entre validez local, anchura física y estabilidad estadística. Las configuraciones con 6 bins también son aceptables, pero presentan anchuras algo mayores.

Para `chi_eff`, no se obtiene una configuración que satisfaga completamente los criterios fuertes. Por ello se selecciona la mejor configuración relajada disponible:

* taxonomía: `difficulty`;
* intervalo: `asymmetric`;
* número de bins: 4;
* cobertura global: 0.900233;
* anchura física mediana: 0.651478;
* `n_bad_bins_p005 = 1`;
* `bad_bin_fraction = 0.25`;
* `n_bins_under_2sigma = 0`;
* `n_bins_outside_2sigma = 0`;
* `all_bins_within_2sigma = True`.

Este resultado indica que, aunque la cobertura global y la banda local de 2σ son aceptables, existe un bin identificado como estadísticamente problemático por el test binomial de infra-cobertura. Por tanto, `chi_eff` debe reportarse como el parámetro más difícil del análisis. La configuración seleccionada no debe interpretarse como localmente perfecta, sino como la mejor opción relajada bajo los criterios definidos.

En conjunto, los resultados muestran que el uso de la taxonomía basada en dificultad (`difficulty`) es preferible para los tres parámetros seleccionados. Para las masas, se obtienen configuraciones con validez local estricta. Para `chi_eff`, la cobertura global es correcta, pero la validez local sigue siendo el principal factor limitante.


**Nota:** la política de selección prioriza la ausencia de infra-cobertura local sobre la maximización del número de bins. Por tanto, una configuración con menos bins puede ser preferible si ofrece intervalos más estables y localmente válidos.

## Plots

In [ ]:
plot_labels = {
    "chirp_mass": r"$\mathcal{M}$",
    "total_mass": r"$M_{\mathrm{tot}}$",
    "chi_eff": r"$\chi_{\mathrm{eff}}$",
}

plot_colors = {
    "symmetric": "blueviolet",
    "asymmetric": "indigo",
}


def plot_global_coverage_vs_bins(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    confidence_level,
    plot_labels,
    plot_colors=None,
):
    df_label = summary_df[summary_df["label"] == label]

    fig, axes = plt.subplots(
        1,
        len(taxonomy_modes),
        figsize=(14, 5),
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    for ax, taxonomy_mode in zip(axes, taxonomy_modes):
        df_tax = df_label[df_label["taxonomy_mode"] == taxonomy_mode]

        if df_tax.empty:
            ax.set_title(f"{taxonomy_mode} (no data)")
            continue

        # Global tolerance is constant for this label.
        low_1 = df_tax["global_tol_1sigma_low"].iloc[0]
        high_1 = df_tax["global_tol_1sigma_high"].iloc[0]
        low_2 = df_tax["global_tol_2sigma_low"].iloc[0]
        high_2 = df_tax["global_tol_2sigma_high"].iloc[0]
        low_3 = df_tax["global_tol_3sigma_low"].iloc[0]
        high_3 = df_tax["global_tol_3sigma_high"].iloc[0]

        ax.axhspan(low_1, high_1, alpha=0.40, label=r"$1\sigma$")
        ax.axhspan(low_2, high_2, alpha=0.35, label=r"$2\sigma$")
        ax.axhspan(low_3, high_3, alpha=0.30, label=r"$3\sigma$")

        ax.axhline(
            confidence_level,
            color="black",
            linestyle="--",
            linewidth=1,
            label=rf"C.L. = {int(confidence_level * 100)}%",
        )

        for interval_mode in interval_modes:
            df_mode = (
                df_tax[df_tax["interval_mode"] == interval_mode]
                .sort_values("n_bins")
            )

            color = None if plot_colors is None else plot_colors.get(interval_mode)

            ax.plot(
                df_mode["n_bins"],
                df_mode["global_coverage"],
                marker="o",
                label=interval_mode,
                color=color,
            )

        ax.set_xticks(sorted(df_tax["n_bins"].unique()))
        ax.set_xlabel(r"$n_{\mathrm{bins}}$", fontsize=13)
        ax.set_title(taxonomy_mode)
        ax.grid(alpha=0.25)

    axes[0].set_ylabel("Global coverage", fontsize=13)

    fig.suptitle(f"Coverage vs bins ({plot_labels[label]})", fontsize=16)

    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        loc="upper left",
        bbox_to_anchor=(0.82, 1.08),
        ncol=2,
    )

    plt.show()

In [ ]:
for label in label_names:
    plot_global_coverage_vs_bins(
        summary_df=summary_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        confidence_level=confidence_level,
        plot_labels=plot_labels,
        plot_colors=plot_colors,
    )

In [ ]:
def plot_width_vs_bins(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    plot_labels,
    width_column="global_median_width_std",
    plot_colors=None,
):
    df_label = summary_df[summary_df["label"] == label]

    fig, axes = plt.subplots(
        1,
        len(taxonomy_modes),
        figsize=(14, 5),
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    for ax, taxonomy_mode in zip(axes, taxonomy_modes):
        df_tax = df_label[df_label["taxonomy_mode"] == taxonomy_mode]

        for interval_mode in interval_modes:
            df_mode = (
                df_tax[df_tax["interval_mode"] == interval_mode]
                .sort_values("n_bins")
            )

            color = None if plot_colors is None else plot_colors.get(interval_mode)

            ax.plot(
                df_mode["n_bins"],
                df_mode[width_column],
                marker="o",
                label=interval_mode,
                color=color,
            )

        ax.set_title(taxonomy_mode)
        ax.set_xlabel(r"$n_{\mathrm{bins}}$", fontsize=13)
        ax.set_xticks(sorted(df_tax["n_bins"].unique()))
        ax.grid(alpha=0.25)

    axes[0].set_ylabel(width_column, fontsize=13)
    fig.suptitle(f"Interval width vs bins ({plot_labels[label]})", fontsize=16)

    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.08),
        ncol=2,
    )

    plt.show()

In [ ]:
for label in label_names:
    plot_width_vs_bins(
        summary_df=summary_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        plot_labels=plot_labels,
        width_column="global_median_width_phys",
        plot_colors=plot_colors,
    )

In [ ]:
def plot_coverage_width_tradeoff(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    confidence_level,
    plot_labels,
    width_column="global_median_width_std",
):
    df_label = summary_df[summary_df["label"] == label]

    scatter_color = {

    }

    plt.figure(figsize=(16, 10))

    for taxonomy_mode in taxonomy_modes:
        for interval_mode in interval_modes:
            df_mode = df_label[
                (df_label["taxonomy_mode"] == taxonomy_mode) &
                (df_label["interval_mode"] == interval_mode)
            ]

            plt.scatter(
                df_mode[width_column],
                df_mode["global_coverage"],
                s=350,
                label=f"{taxonomy_mode}-{interval_mode}",
            )

            for _, row in df_mode.iterrows():
                plt.text(
                    row[width_column],
                    row["global_coverage"],
                    str(row["n_bins"]),
                    fontsize=14,
                    ha="center",
                    va="center",
                    color="white",
                )

    plt.axhline(confidence_level, color="black", linestyle="--", linewidth=1)
    plt.xlabel(width_column)
    plt.ylabel("Global coverage")
    plt.title(f"Coverage-efficiency tradeoff ({plot_labels[label]})", fontsize=18)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=14)
    plt.show()

In [ ]:
for label in label_names:
    plot_coverage_width_tradeoff(
        summary_df=candidate_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        confidence_level=confidence_level,
        plot_labels=plot_labels,
        width_column="global_median_width_phys",
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_coverage_and_width_per_bin(
    all_results,
    key,
    label_idx,
    label_names,
    plot_labels,
    confidence_level,
    width_stat="mean",          # "mean" or "median"
    width_space="std",          # "std" or "phys"
    y_std=None,                 # required if width_space="phys"
    coverage_ylim=(0.85, 0.95),
    width_ylim=None,
    annotate_counts=True,
):
    result = all_results[key]
    metrics = result.metrics

    label = label_names[label_idx]
    title_label = plot_labels[label] if label in plot_labels else label

    coverage_per_bin = metrics["coverage_per_bin"][:, label_idx]
    counts_per_bin = metrics["counts_per_bin"][:, label_idx]
    bin_tol = metrics["bin_tolerance_normal"]

    n_bins = len(coverage_per_bin)
    x = np.arange(n_bins)

    # ------------------------------------------------------------------
    # Width per bin
    # ------------------------------------------------------------------
    width_key = f"{width_stat}_width_per_bin"

    if width_key in metrics:
        width_per_bin = metrics[width_key][:, label_idx].astype(float)
    else:
        # Fallback: compute from test intervals
        bin_indices_test = result.bin_indices_test
        widths = result.upper - result.lower  # (n_test, n_labels)

        width_per_bin = np.full(n_bins, np.nan)

        for b in range(n_bins):
            mask = bin_indices_test[:, label_idx] == b

            if np.sum(mask) == 0:
                continue

            if width_stat == "mean":
                width_per_bin[b] = np.mean(widths[mask, label_idx])
            elif width_stat == "median":
                width_per_bin[b] = np.median(widths[mask, label_idx])
            else:
                raise ValueError("width_stat must be 'mean' or 'median'.")

    if width_stat not in ["mean", "median"]:
        raise ValueError("width_stat must be 'mean' or 'median'.")

    if width_space == "phys":
        if y_std is None:
            raise ValueError("y_std must be provided when width_space='phys'.")

        width_per_bin = width_per_bin * y_std[label_idx]
        width_ylabel = f"{width_stat.capitalize()} interval width [{label}]"

    elif width_space == "std":
        width_ylabel = f"{width_stat.capitalize()} interval width [standardized]"

    else:
        raise ValueError("width_space must be 'std' or 'phys'.")

    # ------------------------------------------------------------------
    # Coverage bands
    # ------------------------------------------------------------------
    low_1 = bin_tol["1sigma_low"][:, label_idx]
    low_2 = bin_tol["2sigma_low"][:, label_idx]
    low_3 = bin_tol["3sigma_low"][:, label_idx]

    high_1 = bin_tol["1sigma_high"][:, label_idx]
    high_2 = bin_tol["2sigma_high"][:, label_idx]
    high_3 = bin_tol["3sigma_high"][:, label_idx]

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(9.5, 5.2))

    band_color = "tab:red"

    ax1.fill_between(
        x, low_3, high_3,
        color=band_color,
        alpha=0.10,
        label=r"Nominal 3$\sigma$",
        zorder=1,
    )

    ax1.fill_between(
        x, low_2, high_2,
        color=band_color,
        alpha=0.15,
        label=r"Nominal 2$\sigma$",
        zorder=2,
    )

    ax1.fill_between(
        x, low_1, high_1,
        color=band_color,
        alpha=0.20,
        label=r"Nominal 1$\sigma$",
        zorder=3,
    )

    ax1.plot(
        x,
        coverage_per_bin,
        marker="o",
        color="maroon",
        label="Empirical coverage",
        zorder=4,
    )

    ax1.axhline(
        confidence_level,
        linestyle="--",
        alpha=0.6,
        linewidth=1.0,
        color="black",
        label=rf"C.L. = {confidence_level}",
    )

    if annotate_counts:
        for i, n in enumerate(counts_per_bin):
            if np.isfinite(coverage_per_bin[i]):
                ax1.text(
                    i,
                    coverage_per_bin[i] + 0.005,
                    str(int(n)),
                    ha="center",
                    fontsize=9,
                    color="black",
                )

    ax1.set_xlabel("Bin index")
    ax1.set_ylabel("Coverage")
    ax1.set_ylim(*coverage_ylim)
    ax1.grid(alpha=0.25)

    # ------------------------------------------------------------------
    # Right axis: interval width
    # ------------------------------------------------------------------
    ax2 = ax1.twinx()

    ax2.plot(
        x,
        width_per_bin,
        marker="s",
        linestyle="-",
        color="tab:blue",
        label=f"{width_stat} interval width",
        zorder=5,
    )

    ax2.set_ylabel(width_ylabel)

    if width_ylim is not None:
        ax2.set_ylim(*width_ylim)

    ax1.set_title(
        f"Coverage and interval width per bin | {title_label} | {key}"
    )

    # Combine legends
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        ncols=2,
        loc="lower right",
    )

    plt.tight_layout()
    plt.show()

    return width_per_bin

## Final candidate configuration

In [ ]:
def inspect_configuration(
    all_results,
    key,
    label_idx,
    label_names,
):
    result = all_results[key]
    metrics = result.metrics
    label = label_names[label_idx]

    print("Configuration:", key)
    print("Label:", label)

    print("\nCounts per bin:")
    print(metrics["counts_per_bin"][:, label_idx])

    print("\nCoverage per bin:")
    print(np.round(metrics["coverage_per_bin"][:, label_idx], 3))

    print("\nCovered count per bin:")
    print(metrics["covered_count_per_bin"][:, label_idx])

    print("\nBin undercoverage p-values:")
    print(np.round(metrics["bin_undercoverage_pvalue"][:, label_idx], 4))

    print("\nLower miss rate per bin:")
    print(np.round(metrics["lower_miss_rate_per_bin"][:, label_idx], 3))

    print("\nUpper miss rate per bin:")
    print(np.round(metrics["upper_miss_rate_per_bin"][:, label_idx], 3))

    print("\nInterval offsets per bin:")
    print(np.round(result.intervals[label_idx], 4))

    print("\nCalibrator bin counts:")
    print(result.calibrator.bin_counts_[label_idx])

    if hasattr(result.calibrator, "quantile_indices_"):
        print("\nQuantile indices:")
        print(result.calibrator.quantile_indices_[label_idx])

In [ ]:
for row in final_by_label.itertuples():
    key = (row.taxonomy_mode, row.interval_mode, row.n_bins)
    label_idx = int(row.label_index)
    label = label_names[label_idx]

    print("=" * 80)
    print(f"Selected configuration for label: {label}")
    print(f"key = {key}")
    print("=" * 80)

    inspect_configuration(
        all_results=all_results,
        key=key,
        label_idx=label_idx,
        label_names=label_names,
    )

    _ = plot_coverage_and_width_per_bin(
        all_results=all_results,
        key=key,
        label_idx=label_idx,
        label_names=label_names,
        plot_labels=plot_labels,
        confidence_level=confidence_level,
        width_stat="median",
        width_space="phys",
        y_std=y_std,
    )

## Final Report

In [ ]:
report_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_median_width_phys",
    "relative_width_excess",
    "min_coverage_per_bin",
    "max_undercoverage_gap",
    "n_bins_under_2sigma",
    "n_bins_outside_2sigma",
    "min_count_per_bin",
]

final_report_df = final_by_label[report_cols].copy()
final_report_df

In [ ]:
final_config_path = mondrian_results_dir / (
    prediction_file.stem + f"_mondrian_final_configs_{mode}.csv"
)

final_by_label[display_cols].to_csv(final_config_path, index=False)
print("Saved:", final_config_path)

In [ ]:
for row in final_by_label.itertuples():
    print("=" * 80)
    print(f"Label: {row.label}")
    print(f"Selection policy: {row.selection_policy}")
    print(f"Config: taxonomy={row.taxonomy_mode}, interval={row.interval_mode}, n_bins={row.n_bins}")
    print(f"Global coverage: {row.global_coverage:.4f}")
    print(f"Median physical width: {row.global_median_width_phys:.4f}")
    print(f"n_bad_bins_p005: {row.n_bad_bins_p005}")
    print(f"bad_bin_fraction: {row.bad_bin_fraction:.3f}")
    print(f"n_bins_under_2sigma: {row.n_bins_under_2sigma}")
    print(f"n_bins_outside_2sigma: {row.n_bins_outside_2sigma}")
    print(f"max_undercoverage_gap: {row.max_undercoverage_gap:.4f}")

    if row.selection_policy == "relaxed_best_available":
        print(
            "NOTE: No strong locally valid configuration was selected for this label. "
            "This result should be reported as the best available relaxed candidate."
        )

- chirp_mass:

  Mejor candidato razonable: difficulty symmetric/asymmetric n_bins=4.
  Aunque prediction n_bins=4 aparece arriba por cobertura local, difficulty n_bins=4 da anchura física bastante menor (~13.9 frente a ~16.0).

- total_mass:

  Mejor candidato razonable: difficulty asymmetric n_bins=4.
  Tiene cobertura global buena, sin bins malos, poca cola desbalanceada y anchura física ~28.0.

- chi_eff:

  Es más delicado.
  Todas las mejores configs tienen algún bin malo.
  Candidatos razonables: difficulty asymmetric n_bins=4 o difficulty symmetric n_bins=4.
  Pero no lo vendería como localmente perfecto.

## (Optional) Inspecting a Configuration

In [ ]:
key = ("difficulty", "asymmetric", 12)
inspect_configuration(
    all_results=all_results,
    key=key,
    label_idx=0,
    label_names=label_names,
)